In [ ]:
import os
import sys
import time
import datetime

from termcolor import colored, cprint

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
from os import listdir

import numpy as np
import pandas as pd  # Import Pandas and Numpy to create databases
import tensorflow as tf  # Import tensorflow to do tensor manipulations
from abyss.helpers import metrics_viewer
from abyss.models import create_decoder
from abyss.glob import glob
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
from sklearn.metrics import roc_curve, auc

In [ ]:
phenotypes_labels = {
    "bc": "Breast Cancer",
    "cad": "Coronary Artery Disease",
    "ms": "Multiple Sclerosis",
    "t2d": "Type 2 Diabetes",
    "pc": "Prostate Cancer",
    "ibd": "Inflammatory Bowel Disease"
}

scenario_labels = {
    "no_cov": "no covariates",
    "abyss": "Abyss correction",
    "cov": "PCA correction",
    "abyss_cov": "Abyss + PCA as covariates"
}

In [ ]:
disease = "ms"

In [ ]:
p_max = 1000
p_min = 6

In [ ]:
path_prs_folder = f"{glob.PATH_PHENO}/PRS/"
path_prs = f"{path_prs_folder}{disease}/"
list_of_bins = os.listdir(path_prs)

In [ ]:
sns.set_palette("rocket")
accuracy = []
tool_fprs = []
tool_tprs = []
tools = []
for bin in list_of_bins[0:1]:
    bin = "bin_0"
    path_bin = path_prs + bin
    arr = [f for f in os.listdir(path_bin) if f.startswith("prs")]
    arr = [f for f in arr if f.split("vals_")[1].split(".pkl")[0] == f"{p_min}_{p_max}"]
    
    for prs in arr:
        correl = float(prs.split("_corr_")[1].split("_")[0])
        tools.append(prs.split("_corr_")[0].split("prs_")[1])
        temp = pd.read_pickle(path_bin + f"/{prs}")
        true_labels = temp['phenotype']
        prs_scaled_values = temp['PRS']
    
        # Calculate the ROC curve and AUC
        fpr, tpr, _ = roc_curve(true_labels, prs_scaled_values)
        roc_auc = auc(fpr, tpr)
        tool_fprs.append(fpr)
        tool_tprs.append(tpr)
    
# Create a figure and axis
plt.figure(figsize=(8, 6))

# Get the Seaborn color palette for the number of tools you have
palette = sns.color_palette("rocket", n_colors=len(tools))

# Plot ROC curve for each PRS correction tool

# Extract 4 specific colors from the "rocket" palette
specific_colors = sns.color_palette("rocket", 4)

# Define a color mapping for your tools
color_dict = {
    'Abyss': specific_colors[0],
    'No covariates': specific_colors[3],
    'PCA': specific_colors[2],
    'Abyss + PCA': specific_colors[1]
}

# Create a mapping dictionary for tool names
tool_mapping = {
    'abyss': 'Abyss',
    'no_cov': 'No covariates',
    'cov': 'PCA',
    'abyss_cov': 'Abyss + PCA'
}

# Map the tools array using the dictionary
mapped_tools = [tool_mapping[tool] for tool in tools]

# Sort order for the tools
sort_order = ['Abyss', 'Abyss + PCA', 'PCA', 'No covariates']

# Combine tool names, FPRs, and TPRs into a single list of tuples
combined_data = list(zip(mapped_tools, tool_fprs, tool_tprs))

# Sort the combined data based on the sort order
sorted_combined_data = sorted(combined_data, key=lambda x: sort_order.index(x[0]))

# Create a figure and axis
plt.figure(figsize=(8, 6))

# Initialize variables to track the highest AUC and corresponding tool
highest_auc = 0
highest_auc_tool = ""

# Iterate through the tools to find the one with the highest AUC
for tool_name, tool_fpr, tool_tpr in sorted_combined_data:
    roc_auc = auc(tool_fpr, tool_tpr)
    if roc_auc > highest_auc:
        highest_auc = roc_auc
        highest_auc_tool = tool_name

# Format the additional legend entry
high_auc_legend = f"Highest AUC = {highest_auc_tool} with {highest_auc:.4f} "


# Plot the sorted data
for tool_name, tool_fpr, tool_tpr in sorted_combined_data:
    roc_auc = auc(tool_fpr, tool_tpr)
    plt.plot(tool_fpr, tool_tpr, lw=2, label=f'{tool_name} (AUC = {roc_auc:.4f})', color=color_dict[tool_name])

# Plot the diagonal line representing random guessing
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')

# Set labels and title
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
p_value_cutoff_min = f"$10^{{-{p_min}}}$"
plt.title(f"ROC Curve of {phenotypes_labels[disease]} with P value cutoff = {p_value_cutoff_min}")

# Add the main legend
plt.legend(loc='lower right')

highest_auc_tool_color = color_dict[highest_auc_tool]
# Add the additional legend for the highest AUC inside the plot at the top left corner
plt.text(0.05, 0.95, high_auc_legend, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', 
         bbox=dict(boxstyle="round,pad=0.5", edgecolor=highest_auc_tool_color, facecolor='white', alpha=1))

plt.show()

In [ ]:
temp

In [ ]:
# Assuming prs_scaled_values is your array of PRS
plt.hist(prs_scaled_values, bins=30)  # You can adjust the number of bins
plt.xlabel('PRS')
plt.ylabel('Frequency')
plt.title('Histogram of PRS')
plt.show()

# Graph the AUC per bin per phenotype

In [ ]:
diseases = ['bc','cad','ms']
diseases = [f"{disease}"]

tools = []
AUCs = []
bins = []
diseases_array = []
    
# Filter out "bin_0" from list_of_bins
list_of_bins = [bin for bin in list_of_bins if bin != "bin_0"]
for disease in diseases:
    for bin in list_of_bins:
        path_prs = f"{path_prs_folder}{disease}/"
        path_bin = path_prs + bin
        arr = [f for f in os.listdir(path_bin) if f.startswith("prs")]
        arr = [f for f in arr if f.split("vals_")[1].split(".pkl")[0] == f"{p_min}_{p_max}"]
        
        for prs in arr:
            correl = float(prs.split("_corr_")[1].split("_")[0])
            tools.append(prs.split("_corr_")[0].split("prs_")[1])
            bins.append(int(bin.split("_")[-1]))
            diseases_array.append(disease)
            temp = pd.read_pickle(path_bin + f"/{prs}")
            true_labels = temp['phenotype']
            prs_scaled_values = temp['PRS']
            # Calculate the ROC curve and AUC
            fpr, tpr, _ = roc_curve(true_labels, prs_scaled_values)
            roc_auc = auc(fpr, tpr)
            AUCs.append(roc_auc)

In [ ]:
df = pd.DataFrame({
    'Tool': tools,
    'AUC': AUCs,
    'Bin': bins,
    'Disease': diseases_array
})

In [ ]:
temps = []
for dis in diseases:
    temp = df[df['Disease'] == dis]
    AUCss = list(temp['AUC'])
    # Calculate the mean of the AUCs
    mean_auc = sum(AUCss) / len(AUCss)
    # Adjust AUCs by subtracting the mean
    adjusted_AUCs = [auc - mean_auc for auc in AUCss]
    temp['AUC_adjusted'] = adjusted_AUCs
    temps.append(temp)

In [ ]:
df = pd.concat(temps, ignore_index=True, axis=0)

In [ ]:
df['AUC_adjusted'] = df['AUC_adjusted'] + abs(df['AUC_adjusted'].min())

In [ ]:
# Plotting
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='Bin', y='AUC_adjusted', hue='Tool', palette='rocket')
plt.title('AUC Adjusted by Tool and Bin')
plt.xlabel('Bin')
plt.ylabel('AUC Adjusted')
plt.legend(title='Tool', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

plt.show()

In [ ]:
unique_bins = np.unique(bins)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Load Seaborn color palette
sns.set_palette("rocket")

# Separate AUC values based on tools and phenotypes
unique_tools = np.unique(tools)
unique_phenotypes = np.unique(diseases_array)

# Create a figure and axis for the plot
fig, ax = plt.subplots(figsize=(10, 6))

# Get the current color palette
palette = sns.color_palette("rocket")
# Select specific colors from the palette
colors = [palette[0], palette[2], palette[3], palette[5]]

# Initialize lists to store the maximum AUC and corresponding tool/phenotype for each bin
max_AUCs = []

# Iterate through bins to find the maximum AUC and its corresponding tool/phenotype
for bin_idx in unique_bins:
    max_bin_AUC = 0
    max_bin_tool_phenotype = None
    max_bin_color = None  # Store the color for each bin
    
    for i, tool in enumerate(unique_tools):
        for j, phenotype in enumerate(unique_phenotypes):
            # Filter AUC values for the current tool, phenotype, and bin
            filter_mask = np.logical_and.reduce((np.array(tools) == tool, np.array(diseases_array) == phenotype, np.array(bins) == bin_idx))
            filtered_AUCs = np.array(AUCs)[filter_mask]
            
            if len(filtered_AUCs) > 0:
                bin_max_AUC = np.max(filtered_AUCs)
                if bin_max_AUC > max_bin_AUC:
                    max_bin_AUC = bin_max_AUC
                    max_bin_tool_phenotype = f'{tool} - {phenotype}'
                    max_bin_color = colors[i]
    
    max_AUCs.append(max_bin_AUC)
    
    # Plot each bar with a specific color
    ax.bar(bin_idx, max_bin_AUC, color=max_bin_color)

bins_init = []
AFs_init = []

path_sim = glob.PATH_GENO+f"/03_macro_similar_AF"
for bin in os.listdir(path_sim):
    bins_init.append(int(bin.split('_')[1]))
    path_AF = path_sim + f"/{bin}"
    AFs_init.append(np.round(float([f for f in os.listdir(path_AF) if f.startswith("AF")][0].split('AF_')[1].split('.pkl')[0].split("_")[0]),2))
mapping = dict(zip(bins_init, AFs_init))

# Reorder AFs_init according to unique_bins
AFs_reordered = [mapping[bin] for bin in unique_bins]

AFs_reordered

# Set x-axis labels and tick positions
ax.set_xticks(unique_bins)  # Use numerical values for ticks
ax.set_xticklabels(AFs_reordered, rotation=45, ha="right")  # Use AF ranges as labels, with a slight rotation for readability

# Set labels and title
ax.set_xlabel('Allele Frequency Bins')
ax.set_ylabel('Highest AUC')
ax.set_title(f"Highest AUC per Bin for {phenotypes_labels[disease]} with P value cutoff = {p_value_cutoff_min}")

# Create a custom legend with labels matching the Seaborn palette
legend_labels = {
    'b': 'Abyss',
    'g': 'Abyss + PCA',
    'r': 'PCA',
    'c': 'No covariates'
}

legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[i], markersize=10, label=label) for i, label in enumerate(legend_labels.values())]

# Add the legend with custom handles
ax.legend(handles=legend_handles, title="Legend", loc='upper left')

# Show the plot
plt.grid(axis='y')
plt.tight_layout()
plt.show()
